In [1]:
from __future__ import annotations

# =========================================================
# PS S6E4 - 025 family + synthetic-threshold-min
#
# Suggested experiment id:
#   exp_20260420_053_lgb025_threshold_min_from_family
#
# Purpose:
# - Keep 025 family skeleton as faithfully as possible
# - Add only a minimal synthetic-threshold block derived from the
#   original-data discussion
# - Use 046b-style 3-stage bias search instead of 025 simple class-weight bias
#
# References used for reconstruction:
# - 025 notebook structure: digit FE + category remap + multiclass TargetEncoder + LGBM fileciteturn10file0
# - 046b bias search: 3-stage log-prob-space bias refine fileciteturn10file1
# =========================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path
import itertools

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier, log_evaluation, early_stopping
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

In [3]:
# ---------------------------------------------------------
# CFG
# ---------------------------------------------------------
class CFG:
    EXP_ID = "exp_20260420_053_lgb025_threshold_min_from_family"
    EXP_NAME = "lgb_digit_te_threshold_min_from_family"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 2026
    N_FOLDS = 5
    NUM_CLASSES = 3

    LGB_PARAMS = {
        "n_estimators": 6000,
        "boosting_type": "gbdt",
        "max_depth": 4,
        "num_leaves": 32,
        "learning_rate": 0.05,
        "feature_fraction": 0.6,
        "bagging_fraction": 0.7,
        "bagging_freq": 1,
        "lambda_l1": 10,
        "lambda_l2": 10,
        "min_child_samples": 12,
        "random_state": 2026,
        "n_jobs": -1,
        "max_bin": 15000,
        "verbosity": -1,
        "subsample": 0.5,
        "subsample_for_bin": 100000,
        "subsample_freq": 1,
    }

In [4]:
# ---------------------------------------------------------
# utility
# ---------------------------------------------------------
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)


def balanced_acc_score_mc(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    c = len(np.unique(y_true))
    acc = 0.0
    for i in range(c):
        acc += np.sum((y_true == i) & (y_pred == i)) / np.sum(y_true == i) / c
    return acc


def lgb_eval_metric(y_true, y_pred):
    score = balanced_acc_score_mc(y_true, y_pred)
    return "acc", score, True


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def safe_logit_like(proba, eps=1e-12):
    return np.log(np.clip(proba, eps, 1.0))


def apply_bias(proba, bias):
    logp = safe_logit_like(proba)
    adjusted = logp + np.array(bias, dtype=np.float32).reshape(1, -1)
    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    expv = np.exp(adjusted)
    out = expv / expv.sum(axis=1, keepdims=True)
    return out.astype(np.float32)


def score_bias(proba, y_true, bias):
    adj = apply_bias(proba, bias)
    pred = np.argmax(adj, axis=1)
    return balanced_accuracy_score(y_true, pred)


def run_grid_search(proba, y_true, bias_ranges):
    best_bias = None
    best_score = -1.0

    for b0, b1, b2 in itertools.product(*bias_ranges):
        bias = (b0, b1, b2)
        sc = score_bias(proba, y_true, bias)
        if sc > best_score:
            best_score = sc
            best_bias = bias

    return best_bias, float(best_score)


seed_everything(CFG.SEED)

In [5]:
# ---------------------------------------------------------
# load data
# ---------------------------------------------------------
drop_cols = [CFG.ID_COL]

train = pd.read_csv(CFG.TRAIN_PATH).drop(drop_cols, axis=1)
test = pd.read_csv(CFG.TEST_PATH).drop(drop_cols, axis=1)

print(f"train.shape={train.shape}, test.shape={test.shape}")

target2idx = {v: i for i, v in enumerate(train[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}

train[CFG.TARGET_COL] = train[CFG.TARGET_COL].map(target2idx)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

print("CATS:", CATS)
print("NUMS:", NUMS)
print(train.head())

train.shape=(630000, 20), test.shape=(270000, 19)
CATS: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
  Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  Sunlight_Hours  Wind_Speed_kmh  Crop_Type Crop_Growth_Stage  Season  \
0     Loamy     4.92          32.58            1.01                     3.05          15.01     50.61       725.99            5.90           16.79  Sugarcane            Sowing    Zaid   
1      Clay     7.08          56.61            0.44                     2.00          22.92     67.86       985.66            6.98            3.39      Wheat        Vegetative  Kharif   
2      Clay     5.69          27.71            0.81    

In [6]:
# ---------------------------------------------------------
# digit FE (keep 025 skeleton)
# ---------------------------------------------------------
M = train[NUMS].max()


def add_digit_features(df):
    df = df.copy()

    for c in NUMS:
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = (df[c] // (10 ** k) % 10).astype("int8")

        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

    return df


train = add_digit_features(train)
test = add_digit_features(test)

In [7]:
# ---------------------------------------------------------
# synthetic-threshold-min block (minimal addition)
# ---------------------------------------------------------
def add_threshold_min_features(df):
    df = df.copy()

    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype("int8")
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype("int8")
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype("int8")
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype("int8")

    # discussion-derived simple score features
    high_score = (
        2 * df["soil_lt_25"]
        + 2 * df["rain_lt_300"]
        + 1 * df["temp_gt_30"]
        + 1 * df["wind_gt_10"]
    )
    low_score = (
        2 * (df["Crop_Growth_Stage"].eq("Harvest").astype("int8"))
        + 2 * (df["Crop_Growth_Stage"].eq("Sowing").astype("int8"))
        + 1 * (df["Mulching_Used"].eq("Yes").astype("int8"))
    )
    df["deotte_high_score"] = high_score.astype("int8")
    df["deotte_low_score"] = low_score.astype("int8")
    df["deotte_score"] = (high_score - low_score).astype("int8")

    # light categorical thresholds for TE/remap usage
    df["score_band"] = pd.cut(
        df["deotte_score"],
        bins=[-999, 0, 3, 999],
        labels=["low_band", "mid_band", "high_band"],
        include_lowest=True,
    ).astype(str)

    return df


train = add_threshold_min_features(train)
test = add_threshold_min_features(test)

In [8]:
# ---------------------------------------------------------
# drop constant columns
# ---------------------------------------------------------
DROP = [c for c in test.columns if test[c].nunique() == 1]
print("DROP:", DROP)

train.drop(DROP, axis=1, inplace=True)
test.drop(DROP, axis=1, inplace=True)

DROP: ['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Humidity_digit2', 'Humidity_digit3', 'Sunlight_Hours_digit2', 'Sunlight_Hours_digit3', 'Wind_Speed_kmh_digit2', 'Wind_Speed_kmh_digit3', 'Field_Area_hectare_digit2', 'Field_Area_hectare_digit3', 'Previous_Irrigation_mm_digit3']


In [9]:
# ---------------------------------------------------------
# category remap
# ---------------------------------------------------------
DIGIT_COLS = [c for c in test.columns if "digit" in c]
THRESHOLD_CAT_COLS = [
    "soil_lt_25",
    "temp_gt_30",
    "rain_lt_300",
    "wind_gt_10",
    "deotte_high_score",
    "deotte_low_score",
    "deotte_score",
    "score_band",
]

CATEGORY = CATS + DIGIT_COLS + THRESHOLD_CAT_COLS

for c in CATEGORY:
    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)

    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = CATEGORY + NUMS

print("n_CATEGORY =", len(CATEGORY))
print("n_FEATURES =", len(FEATURES))
print(train[FEATURES + [CFG.TARGET_COL]].head())

n_CATEGORY = 82
n_FEATURES = 93
   Soil_Type  Crop_Type  Crop_Growth_Stage  Season  Irrigation_Type  Water_Source  Mulching_Used  Region  Soil_pH_digit-4  Soil_pH_digit-3  Soil_pH_digit-2  Soil_pH_digit-1  Soil_pH_digit0  \
0          2          0                  3       2                3             3              0       2                0                0                4                1               4   
1          1          4                  2       0                2             1              1       0                0                0                8                2               2   
2          1          1                  2       0                1             0              1       4                1                1                9                0               1   
3          0          4                  1       0                0             1              1       0                1                1                5                0               1   
4       

In [10]:
# ---------------------------------------------------------
# class weight
# ---------------------------------------------------------
unique, counts = np.unique(train[CFG.TARGET_COL].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[CFG.TARGET_COL]])

print("weights_dict =", weights_dict)

weights_dict = {np.int64(0): np.float64(0.5676949153458749), np.int64(1): np.float64(0.8783891180136694), np.int64(2): np.float64(9.995716121662145)}


In [11]:
# ---------------------------------------------------------
# CV main (keep 025 skeleton)
# ---------------------------------------------------------
X = train.drop([CFG.TARGET_COL], axis=1)
y = train[CFG.TARGET_COL]
test_X = test.copy()

oof_raw = np.zeros((len(y), CFG.NUM_CLASSES), dtype=np.float32)
pred_raw = np.zeros((len(test_X), CFG.NUM_CLASSES), dtype=np.float32)

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=42)
fold_scores = []
best_iterations = []
feature_importance_rows = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(f"\nFold {fold}/{CFG.N_FOLDS}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
    train_weights = sample_weights[train_idx]

    te = TargetEncoder(target_type="multiclass", smooth="auto", cv=5, random_state=42)

    X_train_enc = te.fit_transform(X_train[FEATURES], y_train)
    X_val_enc = te.transform(X_val[FEATURES])
    X_test_enc = te.transform(test_X[FEATURES])

    X_train_enc = pd.DataFrame(X_train_enc, index=X_train.index)
    X_val_enc = pd.DataFrame(X_val_enc, index=X_val.index)
    X_test_enc = pd.DataFrame(X_test_enc, index=test_X.index)

    X_train = pd.concat([X_train, X_train_enc], axis=1)
    X_val = pd.concat([X_val, X_val_enc], axis=1)
    X_test = pd.concat([test_X, X_test_enc], axis=1)

    # shared/025-aligned: drop original raw categorical columns only
    X_train = X_train.drop(CATS, axis=1)
    X_val = X_val.drop(CATS, axis=1)
    X_test = X_test.drop(CATS, axis=1)

    model = LGBMClassifier(**CFG.LGB_PARAMS)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        sample_weight=train_weights,
        eval_metric=lgb_eval_metric,
        callbacks=[log_evaluation(100), early_stopping(250)],
    )

    y_pred = model.predict_proba(X_val)
    oof_raw[val_idx] = y_pred
    pred_raw += model.predict_proba(X_test) / CFG.N_FOLDS

    fold_ba = balanced_acc_score_mc(y_val.values, y_pred)
    fold_scores.append(float(fold_ba))
    best_iterations.append(int(model.best_iteration_))
    print(f"fold_ba = {fold_ba:.6f} | best_iteration = {model.best_iteration_}")

    fi = pd.DataFrame({
        "feature": X_train.columns,
        f"gain_f{fold:02d}": model.booster_.feature_importance(importance_type="gain"),
    })
    feature_importance_rows.append(fi)


Fold 1/5
Training until validation scores don't improve for 250 rounds
[100]	valid_0's multi_logloss: 0.0664373	valid_0's acc: 0.975433
[200]	valid_0's multi_logloss: 0.0582308	valid_0's acc: 0.977359
[300]	valid_0's multi_logloss: 0.0549108	valid_0's acc: 0.977635
[400]	valid_0's multi_logloss: 0.0529654	valid_0's acc: 0.978095
[500]	valid_0's multi_logloss: 0.0516599	valid_0's acc: 0.978072
[600]	valid_0's multi_logloss: 0.0506919	valid_0's acc: 0.977863
[700]	valid_0's multi_logloss: 0.0499598	valid_0's acc: 0.977568
Early stopping, best iteration is:
[457]	valid_0's multi_logloss: 0.0521662	valid_0's acc: 0.97829
fold_ba = 0.978290 | best_iteration = 457

Fold 2/5
Training until validation scores don't improve for 250 rounds
[100]	valid_0's multi_logloss: 0.0659465	valid_0's acc: 0.978434
[200]	valid_0's multi_logloss: 0.0579607	valid_0's acc: 0.979475
[300]	valid_0's multi_logloss: 0.054909	valid_0's acc: 0.979704
[400]	valid_0's multi_logloss: 0.0531326	valid_0's acc: 0.979854
[

In [12]:
# ---------------------------------------------------------
# raw CV
# ---------------------------------------------------------
raw_cv_ba = balanced_acc_score_mc(y.values, oof_raw)
print(f"raw_cv_ba = {raw_cv_ba:.6f}")

raw_cv_ba = 0.979083


In [13]:
# ---------------------------------------------------------
# 046b-style 3-stage bias search
# ---------------------------------------------------------
# Stage 1: coarse
coarse_vals = np.arange(-1.0, 1.01, 0.1)
best_bias_1, best_score_1 = run_grid_search(
    oof_raw, y,
    [coarse_vals, coarse_vals, coarse_vals]
)
print("stage1 best_bias:", best_bias_1, "score:", best_score_1)

# Stage 2: local refine
b0, b1, b2 = best_bias_1
local_vals_0 = np.arange(b0 - 0.10, b0 + 0.1001, 0.02)
local_vals_1 = np.arange(b1 - 0.10, b1 + 0.1001, 0.02)
local_vals_2 = np.arange(b2 - 0.10, b2 + 0.1001, 0.02)

best_bias_2, best_score_2 = run_grid_search(
    oof_raw, y,
    [local_vals_0, local_vals_1, local_vals_2]
)
print("stage2 best_bias:", best_bias_2, "score:", best_score_2)

# Stage 3: ultra-local refine
b0, b1, b2 = best_bias_2
ultra_vals_0 = np.arange(b0 - 0.02, b0 + 0.0201, 0.005)
ultra_vals_1 = np.arange(b1 - 0.02, b1 + 0.0201, 0.005)
ultra_vals_2 = np.arange(b2 - 0.02, b2 + 0.0201, 0.005)

best_bias_3, best_score_3 = run_grid_search(
    oof_raw, y,
    [ultra_vals_0, ultra_vals_1, ultra_vals_2]
)
print("stage3 best_bias:", best_bias_3, "score:", best_score_3)

final_bias = best_bias_3
final_cv = best_score_3

oof_biased = apply_bias(oof_raw, final_bias)
pred_biased = apply_bias(pred_raw, final_bias)

stage1 best_bias: (np.float64(-1.0), np.float64(-1.0), np.float64(-0.6000000000000001)) score: 0.9791679817212592
stage2 best_bias: (np.float64(-1.04), np.float64(-1.04), np.float64(-0.7000000000000001)) score: 0.9791923044684244
stage3 best_bias: (np.float64(-1.06), np.float64(-1.0550000000000002), np.float64(-0.7050000000000001)) score: 0.9792028362618709


In [14]:
# ---------------------------------------------------------
# final prediction / save
# ---------------------------------------------------------
final_test_preds = np.argmax(pred_biased, axis=1)

submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET_COL] = pd.Series(final_test_preds).map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_raw)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", pred_raw)
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba_biased.npy", oof_biased)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba_biased.npy", pred_biased)
np.save(CFG.SAVE_DIR / "best_class_bias_refined.npy", np.array(final_bias, dtype=np.float32))

feature_cols_df = pd.DataFrame({"feature": X_train.columns.tolist()})
feature_cols_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

if len(feature_importance_rows) > 0:
    fi_merged = feature_importance_rows[0].copy()
    for item in feature_importance_rows[1:]:
        fi_merged = fi_merged.merge(item, on="feature", how="outer")
    gain_cols = [c for c in fi_merged.columns if c.startswith("gain_f")]
    fi_merged[gain_cols] = fi_merged[gain_cols].fillna(0.0)
    fi_merged["gain_mean"] = fi_merged[gain_cols].mean(axis=1)
    fi_merged = fi_merged.sort_values("gain_mean", ascending=False)
    fi_merged.to_csv(CFG.SAVE_DIR / "feature_importance.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "base_parent": "exp_20260409_025_lgb_digit_te_min",
    "variant": "threshold_min_from_family",
    "raw_cv_ba": float(raw_cv_ba),
    "refined_biased_cv": float(final_cv),
    "fold_scores": fold_scores,
    "best_iterations": best_iterations,
    "best_bias": list(map(float, final_bias)),
    "search": {
        "stage1": {"range": [-1.0, 1.0], "step": 0.1},
        "stage2": {"center": list(map(float, best_bias_1)), "width": 0.10, "step": 0.02},
        "stage3": {"center": list(map(float, best_bias_2)), "width": 0.02, "step": 0.005},
    },
    "n_features_before_te": int(len(FEATURES)),
    "n_features_after_te": int(X_train.shape[1]),
    "cats": CATS,
    "nums": NUMS,
    "category_cols": CATEGORY,
    "threshold_cols_added": THRESHOLD_CAT_COLS,
    "lgb_params": CFG.LGB_PARAMS,
}
save_json(cv_result, CFG.SAVE_DIR / "cv_result.json")

summary_df = pd.DataFrame({
    "item": [
        "raw_cv_ba",
        "refined_biased_cv",
        "improvement",
        "n_features_before_te",
        "n_features_after_te",
    ],
    "value": [
        raw_cv_ba,
        final_cv,
        final_cv - raw_cv_ba,
        len(FEATURES),
        X_train.shape[1],
    ],
})
summary_df.to_csv(CFG.SAVE_DIR / "summary.csv", index=False)

print("final_bias:", final_bias)
print("final_cv:", final_cv)
print("saved to:", CFG.SAVE_DIR)
print(submission.head())

final_bias: (np.float64(-1.06), np.float64(-1.0550000000000002), np.float64(-0.7050000000000001))
final_cv: 0.9792028362618709
saved to: /kaggle/working/exp_20260420_053_lgb025_threshold_min_from_family
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
